In [0]:
# Databricks notebook source
# File: config/00_shared_config.py
# Project: MIGRATION-DBX-001
# Purpose: Single source of truth for all paths, constants, schema definitions
#          %run this notebook at the top of every processing notebook
# =============================================================================

In [0]:
# =============================================================================
# 1. SECRET SCOPE + STORAGE CONFIG
# =============================================================================
SECRET_SCOPE     = "migration-kv-scope"
ADLS_ACCOUNT     = dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-account-name")

# Set ADLS OAuth config so Spark can access ADLS via Access Connector MSI
# Note: With Unity Catalog + External Locations, this is often not needed.
# Keeping it as fallback for any direct ADLS path access outside UC tables.
spark.conf.set(
    f"fs.azure.account.auth.type.{ADLS_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{ADLS_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.MsiTokenProvider"
)

In [0]:
# =============================================================================
# 2. LAYER PATHS
# =============================================================================
BRONZE_BASE     = f"abfss://migration-raw@{ADLS_ACCOUNT}.dfs.core.windows.net"
SILVER_BASE     = f"abfss://migration-processed@{ADLS_ACCOUNT}.dfs.core.windows.net/silver"
GOLD_BASE       = f"abfss://migration-processed@{ADLS_ACCOUNT}.dfs.core.windows.net/gold"
QUARANTINE_BASE = f"abfss://migration-quarantine@{ADLS_ACCOUNT}.dfs.core.windows.net"

In [0]:
# =============================================================================
# 3. UNITY CATALOG REFERENCES
# =============================================================================
UC_CATALOG    = "migration_dev"
UC_BRONZE     = f"{UC_CATALOG}.bronze"
UC_SILVER     = f"{UC_CATALOG}.silver"
UC_GOLD       = f"{UC_CATALOG}.gold"
UC_QUARANTINE = f"{UC_CATALOG}.quarantine"

In [0]:
# =============================================================================
# 4. SOURCE TABLE REGISTRY
# Maps logical table name → Bronze folder name → Silver table name
# Single place to update if any name changes
# =============================================================================
TABLE_REGISTRY = {
    "orders": {
        "bronze_folder":  "olist_orders_dataset",
        "silver_table":   f"{UC_SILVER}.orders",
        "pk_cols":        ["order_id"],
        "watermark_col":  "order_purchase_timestamp",
    },
    "order_items": {
        "bronze_folder":  "olist_order_items_dataset",
        "silver_table":   f"{UC_SILVER}.order_items",
        "pk_cols":        ["order_id", "order_item_id"],
        "watermark_col":  "shipping_limit_date",
    },
    "order_payments": {
        "bronze_folder":  "olist_order_payments_dataset",
        "silver_table":   f"{UC_SILVER}.order_payments",
        "pk_cols":        ["order_id", "payment_sequential"],
        "watermark_col":  None,
    },
    "order_reviews": {
        "bronze_folder":  "olist_order_reviews_dataset",
        "silver_table":   f"{UC_SILVER}.order_reviews",
        "pk_cols":        ["review_id", "order_id"],
        "watermark_col":  "review_creation_date",
    },
    "customers": {
        "bronze_folder":  "olist_customers_dataset",
        "silver_table":   f"{UC_SILVER}.customers",
        "pk_cols":        ["customer_id"],
        "watermark_col":  "created_at",
    },
    "products": {
        "bronze_folder":  "olist_products_dataset",
        "silver_table":   f"{UC_SILVER}.products",
        "pk_cols":        ["product_id"],
        "watermark_col":  "created_at",
    },
    "sellers": {
        "bronze_folder":  "olist_sellers_dataset",
        "silver_table":   f"{UC_SILVER}.sellers",
        "pk_cols":        ["seller_id"],
        "watermark_col":  "created_at",
    },
    "geolocation": {
        "bronze_folder":  "olist_geolocation_dataset",
        "silver_table":   f"{UC_SILVER}.geolocation",
        "pk_cols":        ["geolocation_zip_code_prefix"],
        "watermark_col":  "created_at",
    },
    "category_translation": {
        "bronze_folder":  "product_category_name_translation",
        "silver_table":   f"{UC_SILVER}.category_translation",
        "pk_cols":        ["product_category_name_pt"],
        "watermark_col":  None,
    },
}

In [0]:
# =============================================================================
# 5. DELTA WRITE OPTIONS (standard across all Silver + Gold writes)
# =============================================================================
DELTA_WRITE_OPTIONS = {
    "mergeSchema":            "true",    # allow new columns without breaking writes
    "delta.autoOptimize.optimizeWrite": "true",  # auto bin-pack on write
    "delta.autoOptimize.autoCompact":   "true",  # auto compact small files
}

In [0]:
# =============================================================================
# 6. INGESTION DATE HELPER
# Used to read only today's Bronze partition (incremental reads)
# =============================================================================
from datetime import datetime, timezone

def get_ingestion_date(override: str = None) -> str:
    """
    Returns today's date as 'yyyy-MM-dd' for Bronze partition reads.
    Pass override='2026-05-12' to reprocess a specific date (backfill).
    """
    if override:
        return override
    return datetime.now(timezone.utc).strftime("%Y-%m-%d")

In [0]:
# =============================================================================
# 7. LOGGING HELPER
# Structured print with timestamp — visible in Spark UI logs
# =============================================================================
def log(msg: str, level: str = "INFO"):
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    print(f"[{level}] [{ts}] {msg}")

# =============================================================================
# CONFIRM LOAD
# =============================================================================
log("Shared config loaded.")
log(f"ADLS account: {ADLS_ACCOUNT}")
log(f"UC Catalog:   {UC_CATALOG}")
log(f"Tables registered: {list(TABLE_REGISTRY.keys())}")